In [ ]:
# script_extracao_com_flags.py
# OBJETIVO: Extrair dados de TODAS as páginas do namespace 0,
# incluindo uma flag 'is_redirect' para cada uma.

import requests
import json
import os
import time
from datetime import datetime
from collections import defaultdict
from pathlib import Path
from typing import List, Dict, Tuple, Optional, Any
import math

# --- Configurações Globais ---
WIKI_API_URL = "https://wikifavelas.com.br/api.php"
BASE_DIR = Path('../dados/dados_com_flags_redirecionamento') 
BASE_DIR.mkdir(exist_ok=True) 
# --- NOME DO ARQUIVO DE SAÍDA ---
OUTPUT_FILENAME_WITH_FLAGS = 'dados_api_com_flags_e_refs.json' 

# --- Funções de Coleta da API ---

def listar_todas_paginas_ns0(session: requests.Session) -> List[Dict[str, Any]]:
    """
    Lista TODAS as páginas do namespace 0 (incluindo redirects),
    retornando dicionários básicos com id e título.
    """
    print("[FASE 1/X] Listando TODAS as páginas do NS 0 (incluindo redirects)...")
    pages_basic_info = []
    params = {
        'action': 'query', 'format': 'json', 'list': 'allpages',
        'apnamespace': 0, # <-- Namespace principal
        # REMOVIDO: 'apfilterredir': 'nonredirects', # Removemos o filtro de redirect
        'aplimit': 'max', 'continue': ''
    }
    api_call_count = 0
    while True:
        try:
            api_call_count += 1
            print(f"  - Chamada API {api_call_count} para list=allpages...") # Log opcional
            response = session.get(WIKI_API_URL, params=params)
            response.raise_for_status()
            data = response.json()
            batch = data.get('query', {}).get('allpages', [])
            if not batch and 'continue' not in data: # Condição de parada
                 if len(pages_basic_info) == 0 and api_call_count == 1:
                      print("  [AVISO] Nenhuma página encontrada na primeira chamada. Verifique a API ou os parâmetros.")
                 break # Sai do loop se não houver batch E não houver 'continue'

            # Adiciona apenas id e título nesta fase
            pages_basic_info.extend([{'id': p['pageid'], 'titulo': p['title']} for p in batch])

            cont = data.get('continue')
            if not cont:
                break
            params.update(cont)
        except requests.exceptions.RequestException as e:
            print(f"  [ERRO] Falha ao listar páginas: {e}. Tentando novamente em 5s...")
            time.sleep(5)
        except Exception as e:
            print(f"  [ERRO] Erro inesperado ao listar páginas: {e}")
            break # Evita loop infinito em caso de erro persistente

    print(f"  [INFO] Total de {len(pages_basic_info)} páginas encontradas no NS 0.")
    return pages_basic_info

def obter_dados_verbete_otimizado_com_flag(page_id: int, session: requests.Session, tentativas: int = 3) -> Optional[tuple]:
    """
    Coleta dados detalhados (revisões, links, cats) E a flag 'is_redirect'
    com UMA chamada API por página (usando pageid).
    Retorna: (is_redirect_flag, usuarios_dict, referencias_list, categorias_list,
             autor_criacao, data_criacao, data_ultima_edicao) ou None em falha total.
    """
    params = {
        'action': 'query', 'format': 'json', 'pageids': page_id,
        'prop': 'info|revisions|links|categories', 
        'inprop': '',
        'rvlimit': 'max', 'rvprop': 'user|timestamp',
        'pllimit': 'max',
        'plnamespace': 0, 
        'cllimit': 'max'
    }

    for tentativa in range(1, tentativas + 1):
        try:
            response = session.get(WIKI_API_URL, params=params)
            response.raise_for_status()
            data = response.json()
            page_data = next(iter(data.get('query', {}).get('pages', {}).values()), None)

            if not page_data or page_data.get('missing', False): # Verifica se a página existe
                print(f"  [AVISO] Página com ID {page_id} não encontrada ou marcada como 'missing'.")
                # Retorna um padrão indicando que é "não-redirect" e dados vazios
                return (False, {}, [], [], "N/A (Missing)", "N/A (Missing)", "N/A (Missing)") 

            # --- Extrai a flag 'is_redirect' da propriedade 'info' ---
            # A chave 'redirect' só existe no dicionário page_data se for um redirect
            is_redirect_flag = 'redirect' in page_data 

            # Extrai outras informações
            revisoes = page_data.get('revisions', [])
            usuarios = defaultdict(int)
            autor_criacao = "N/A"
            data_criacao = "N/A"
            data_ultima_edicao = "N/A"

            if revisoes: # Processa apenas se houver revisões
                 for rev in revisoes:
                     usuarios[rev.get('user', 'Desconhecido')] += 1
                 # API retorna a mais recente primeiro
                 data_ultima_edicao = revisoes[0].get('timestamp', 'N/A') 
                 # A mais antiga é a última na lista
                 autor_criacao = revisoes[-1].get('user', 'N/A') 
                 data_criacao = revisoes[-1].get('timestamp', 'N/A')
            # Fallback se não houver revisões
            elif 'touched' in page_data: 
                 data_ultima_edicao = page_data.get('touched')
                 data_criacao = data_ultima_edicao # Assume criação = último toque

            # Extrai referências e categorias
            referencias = [link['title'] for link in page_data.get('links', []) if 'title' in link]
            categorias = [cat['title'] for cat in page_data.get('categories', []) if 'title' in cat]

            return (is_redirect_flag, dict(usuarios), referencias, categorias, autor_criacao, data_criacao, data_ultima_edicao)

        except (requests.exceptions.RequestException, json.JSONDecodeError) as e:
            print(f"  [ERRO] Tentativa {tentativa} para page_id {page_id} falhou: {e}. Aguardando...")
            time.sleep(2 * tentativa)
        except Exception as e:
             print(f"  [ERRO INESPERADO] ao processar page_id {page_id}: {e}")
             # Retorna None para indicar falha irrecuperável nesta página específica
             return None

    print(f"  [FALHA] Todas as tentativas para page_id {page_id} falharam.")
    return None # Indica falha completa

# --- Script Principal ---

def main() -> None:
    session = requests.Session()
    # Chama a função que lista TODAS as páginas do NS 0
    paginas_base = listar_todas_paginas_ns0(session) 
    
    paginas_enriquecidas = []
    total_paginas = len(paginas_base)
    
    output_path = BASE_DIR / OUTPUT_FILENAME_WITH_FLAGS # Salva no novo arquivo

    print(f"\n[FASE 2/X] Enriquecendo cada página ({total_paginas}) com dados adicionais e flag redirect...")
    for i, pagina in enumerate(paginas_base, start=1):
        # Mostra o título durante o processamento para acompanhamento
        print(f"  Processando {i}/{total_paginas}: \"{pagina['titulo']}\" (ID: {pagina['id']})") 
        
        # Chama a função modificada que retorna a flag
        dados_coletados = obter_dados_verbete_otimizado_com_flag(pagina['id'], session) 
        
        pagina_completa = pagina.copy() # Começa com id e título
        pagina_completa['link'] = f"https://wikifavelas.com.br/{pagina['titulo'].replace(' ', '_')}" # Adiciona link aqui

        if dados_coletados is not None: # Verifica se a coleta foi bem-sucedida
            # Desempacota os resultados, incluindo a flag
            is_redirect, usuarios, refs, cats, autor, criacao, ultima = dados_coletados
            pagina_completa.update({
                'is_redirect': is_redirect,
                'quantidade_edicoes': sum(usuarios.values()),
                'usuarios_edicoes': usuarios,
                'referencias': refs,
                'categorias': cats,
                'autor_criacao': autor,
                'data_criacao': criacao,
                'data_ultima_edicao': ultima
            })
        else: # Se a coleta falhou completamente
            pagina_completa.update({
                'is_redirect': False,
                'quantidade_edicoes': 0, 'usuarios_edicoes': {}, 'referencias': [],
                'categorias': [], 'autor_criacao': "Falha na Coleta", 
                'data_criacao': "Falha na Coleta", 'data_ultima_edicao': "Falha na Coleta"
            })

        paginas_enriquecidas.append(pagina_completa)
        
        # Salva progresso periodicamente ou no final
        if i % 50 == 0 or i == total_paginas:
            dados_para_salvar = {
                # Atualiza o total para refletir o número de páginas processadas até agora
                'total_paginas_ns0': len(paginas_enriquecidas), 
                'verbetes_completo': paginas_enriquecidas # Mantém nome da chave por compatibilidade
            }
            try:
                with open(output_path, 'w', encoding='utf-8') as f:
                    json.dump(dados_para_salvar, f, ensure_ascii=False, indent=2)
                print(f"  [SALVO] Progresso salvo em '{output_path}' ({i} páginas processadas)")
            except Exception as e:
                 print(f"  [ERRO AO SALVAR] '{output_path}': {e}")


    print(f"\n[FINALIZADO - PASSO 1] Processo de coleta com flags concluído.")
    print(f"Arquivo final salvo em: '{output_path}'")

# --- Execução ---
if __name__ == '__main__':
    main()

[FASE 1/X] Listando TODAS as páginas do NS 0 (incluindo redirects)...
  - Chamada API 1 para list=allpages...
  - Chamada API 2 para list=allpages...
  - Chamada API 3 para list=allpages...
  - Chamada API 4 para list=allpages...
  - Chamada API 5 para list=allpages...
  - Chamada API 6 para list=allpages...
  - Chamada API 7 para list=allpages...
  - Chamada API 8 para list=allpages...
  - Chamada API 9 para list=allpages...
  [INFO] Total de 4034 páginas encontradas no NS 0.

[FASE 2/X] Enriquecendo cada página (4034) com dados adicionais e flag redirect...
  Processando 1/4034: ""A Expansão das Milícias no Rio de Janeiro: uso da força estatal, mercado imobiliário e grupos armados" (Resenha)" (ID: 5967)
  Processando 2/4034: ""Bairro Veneza – Balsas - MA: Dados e Aspectos Socioeconômicos"" (ID: 8544)
  Processando 3/4034: ""Estudo sobre a distribuição das taxas de encarceramento nos estados brasileiros e principais variáveis associadas: Influências socioeconômicas e ideológicas (Rese

In [ ]:
# script: resolve_redirects.py
#
# OBJETIVO: Ler o arquivo JSON que contém a flag 'is_redirect'
#           e consultar a API para mapear cada título de redirecionamento
#           ao seu título de destino final.

import requests
import json
from pathlib import Path
import time
import math

# --- Configurações ---
WIKI_API_URL = "https://wikifavelas.com.br/api.php"
DATA_DIR = Path('../dados/dados_com_flags_redirecionamento')
# Arquivo JSON gerado pelo Passo 1 (extração com flags)
INPUT_FILENAME = 'dados_api_com_flags_e_refs.json'
# Arquivo JSON de saída que conterá o mapa de redirecionamentos
OUTPUT_FILENAME = 'redirect_map.json'

def resolve_redirects_from_flags():
    """
    Função principal para ler dados com flags, resolver redirecionamentos
    via API e salvar o mapa resultante.
    """
    input_path = DATA_DIR / INPUT_FILENAME
    output_path = DATA_DIR / OUTPUT_FILENAME

    # --- 1. Carregar Dados com Flags ---
    if not input_path.exists():
        print(f"ERRO: Arquivo de entrada '{input_path}' não encontrado.")
        print("Certifique-se de ter executado o Passo 1 (extração com flags) primeiro.")
        return

    print(f"Carregando dados com flags de '{input_path}'...")
    try:
        with open(input_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
    except Exception as e:
        print(f"Erro ao carregar ou decodificar o JSON de entrada: {e}")
        return

    all_verbetes = data.get('verbetes_completo', [])
    if not all_verbetes:
        print("ERRO: Lista 'verbetes_completo' não encontrada ou vazia no arquivo de entrada.")
        return

    # --- 2. Identificar Títulos de Redirecionamento ---
    print("Identificando títulos marcados como redirecionamento...")
    redirect_titles = []
    for page in all_verbetes:
         # Verifica se a flag existe e é True
        if page.get('is_redirect') == True and 'titulo' in page: 
            redirect_titles.append(page['titulo'])

    if not redirect_titles:
        print("Nenhuma página marcada como redirecionamento encontrada nos dados.")
        # Salva um mapa vazio para consistência do pipeline
        with open(output_path, 'w', encoding='utf-8') as f:
             json.dump({}, f, ensure_ascii=False, indent=2)
        print(f"Mapa de redirecionamento vazio salvo em '{output_path}'.")
        return

    print(f"Encontrados {len(redirect_titles)} títulos marcados como redirecionamento.")
    print("Iniciando resolução dos alvos via API (pode levar alguns minutos)...")

    # --- 3. Consultar API para Resolver Redirecionamentos em Lotes ---
    redirect_map = {} # Mapa final: redirect_source_title -> final_target_title
    session = requests.Session()
    batch_size = 50 
    max_retries = 5 # Define um número máximo de tentativas por lote para evitar loops infinitos
    initial_wait_time = 5 # Tempo de espera inicial em segundos

    total_batches = math.ceil(len(redirect_titles) / batch_size)
    # Loop FOR principal que itera sobre os lotes
    for i in range(0, len(redirect_titles), batch_size):
        batch_titles = redirect_titles[i:i + batch_size]
        titles_param = "|".join(batch_titles) 

        params = {
            'action': 'query', 'format': 'json', 'titles': titles_param, 'redirects': 1
        }

        current_batch_num = (i // batch_size) + 1
        print(f"  Processando lote {current_batch_num}/{total_batches}...")

        # --- Loop WHILE aninhado para retentativas ---
        retries = 0
        success = False
        while not success and retries < max_retries:
            try:
                # Tenta fazer a chamada API
                response = session.get(WIKI_API_URL, params=params)
                response.raise_for_status() # Verifica erros HTTP
                result_data = response.json()

                # Processa a resposta
                redirects_info = result_data.get('query', {}).get('redirects', [])
                batch_mapped_count = 0
                for redir in redirects_info:
                    if 'from' in redir and 'to' in redir:
                         redirect_map[redir['from']] = redir['to']
                         batch_mapped_count += 1
                    else:
                         print(f"    [AVISO] Mapeamento incompleto recebido: {redir}")
                
                # Tratamento de normalização
                normalized_info = result_data.get('query', {}).get('normalized', [])
                if normalized_info:
                     temp_map_update = {}
                     normalized_lookup = {norm['to']: norm['from'] for norm in normalized_info if norm['from'] in batch_titles}
                     keys_to_update = []
                     # Itera sobre cópia das chaves para poder modificar o dict original
                     for source_in_map in list(redirect_map.keys()): 
                         if source_in_map in normalized_lookup:
                              original_title_from_batch = normalized_lookup[source_in_map]
                              if original_title_from_batch != source_in_map:
                                 temp_map_update[original_title_from_batch] = redirect_map[source_in_map]

                     redirect_map.update(temp_map_update)

                print(f"    -> Lote {current_batch_num} processado com sucesso. {batch_mapped_count} redirecionamentos mapeados.")
                success = True # Marca como sucesso para sair do loop WHILE

            except requests.exceptions.RequestException as e:
                # Se for um erro de rede/conexão...
                retries += 1
                wait_time = initial_wait_time * (2 ** (retries - 1)) # Backoff exponencial
                print(f"  [ERRO] Falha na chamada API para o lote {current_batch_num}: {e}")
                if retries < max_retries:
                    print(f"    Tentativa {retries}/{max_retries}. Tentando novamente em {wait_time} segundos...")
                    time.sleep(wait_time)
                else:
                    print(f"  [FALHA CRÍTICA] Máximo de {max_retries} tentativas atingido para o lote {current_batch_num}. Abortando o processamento deste lote.")
                    # Decide o que fazer: parar o script, logar o lote falho, etc.
                    # Por ora, apenas sai do while e continua o for principal (perdendo o lote)
                    break # Sai do loop WHILE para este lote

            except Exception as e:
                 # Se for outro erro inesperado (JSON malformado, etc.)...
                 print(f"  [ERRO INESPERADO] ao processar lote {current_batch_num}: {e}")
                 print("    Erro não recuperável automaticamente. Abortando o processamento deste lote.")
                 # Considerar parar o script ou logar o erro detalhadamente
                 break # Sai do loop WHILE para este lote
        
        # Pequena pausa entre os lotes (mesmo após sucesso ou falha)
        time.sleep(0.5) 

    # --- 4. Salvar o Mapa de Redirecionamentos ---
    print(f"\nResolução concluída. {len(redirect_map)} redirecionamentos mapeados no total.")
    try:
        with open(output_path, 'w', encoding='utf-8') as f:
            json.dump(redirect_map, f, ensure_ascii=False, indent=2)
        print(f"Mapa de redirecionamento salvo com sucesso em: '{output_path.resolve()}'")
    except Exception as e:
        print(f"ERRO ao salvar o arquivo JSON do mapa: {e}")

# --- Execução ---
if __name__ == '__main__':
    # Certifique-se que o arquivo 'dados_api_com_flags_e_refs.json' existe em '../dados'
    resolve_redirects_from_flags()

Carregando dados com flags de '..\dados\dados_com_flags_redirecionamento\dados_api_com_flags_e_refs.json'...
Identificando títulos marcados como redirecionamento...
Encontrados 740 títulos marcados como redirecionamento.
Iniciando resolução dos alvos via API (pode levar alguns minutos)...
  Processando lote 1/15...
    -> Lote 1 processado com sucesso. 58 redirecionamentos mapeados.
  Processando lote 2/15...
    -> Lote 2 processado com sucesso. 53 redirecionamentos mapeados.
  Processando lote 3/15...
    -> Lote 3 processado com sucesso. 51 redirecionamentos mapeados.
  Processando lote 4/15...
    -> Lote 4 processado com sucesso. 54 redirecionamentos mapeados.
  Processando lote 5/15...
    -> Lote 5 processado com sucesso. 53 redirecionamentos mapeados.
  Processando lote 6/15...
    -> Lote 6 processado com sucesso. 53 redirecionamentos mapeados.
  Processando lote 7/15...
    -> Lote 7 processado com sucesso. 58 redirecionamentos mapeados.
  Processando lote 8/15...
    -> Lote